In [1]:
from collections import Counter

word_freq = {
    "cat": 2,
    "cats": 1,
    "cap": 1
}

splits = {
    "cat": ["c", "##a", "##t"],
    "cats": ["c", "##a", "##t", "##s"],
    "cap": ["c", "##a", "##p"]
}

print("Initial Splits:")
for word, tokens in splits.items():
    print(word, "→", tokens)

Initial Splits:
cat → ['c', '##a', '##t']
cats → ['c', '##a', '##t', '##s']
cap → ['c', '##a', '##p']


In [2]:
token_freq = Counter()
pair_freq = Counter()

for word, freq in word_freq.items():
    tokens = splits[word]

    # Token frequency
    for token in tokens:
        token_freq[token] += freq

    # Pair frequency
    for i in range(len(tokens) - 1):
        pair = (tokens[i], tokens[i + 1])
        pair_freq[pair] += freq

print("Token Frequencies:")
for token, freq in token_freq.items():
    print(token, ":", freq)

print("\nPair Frequencies:")
for pair, freq in pair_freq.items():
    print(pair, ":", freq)

Token Frequencies:
c : 4
##a : 4
##t : 3
##s : 1
##p : 1

Pair Frequencies:
('c', '##a') : 4
('##a', '##t') : 3
('##t', '##s') : 1
('##a', '##p') : 1


In [3]:
scores = {}

for pair, freq in pair_freq.items():
    first = pair[0]
    second = pair[1]

    score = freq / (
        token_freq[first] * token_freq[second]
    )

    scores[pair] = score

print("WordPiece Scores:")

for pair, score in scores.items():
    print(pair, "=", round(score, 4))

WordPiece Scores:
('c', '##a') = 0.25
('##a', '##t') = 0.25
('##t', '##s') = 0.3333
('##a', '##p') = 0.25


In [4]:
best_pair = max(scores, key=scores.get)

print("Best Pair:", best_pair)
print("Best Score:", round(scores[best_pair], 4))

Best Pair: ('##t', '##s')
Best Score: 0.3333


In [5]:
def merge_pair(splits, pair):
    new_token = pair[0] + pair[1].replace("##", "")

    for word in splits:
        tokens = splits[word]
        new_tokens = []
        i = 0

        while i < len(tokens):

            if i < len(tokens) - 1:
                current_pair = (
                    tokens[i],
                    tokens[i + 1]
                )

                if current_pair == pair:
                    new_tokens.append(new_token)
                    i += 2
                    continue

            new_tokens.append(tokens[i])
            i += 1

        splits[word] = new_tokens

    return new_token


new_token = merge_pair(splits, best_pair)

print("New Token:", new_token)

print("\nUpdated Splits:")
for word, tokens in splits.items():
    print(word, "→", tokens)

New Token: ##ts

Updated Splits:
cat → ['c', '##a', '##t']
cats → ['c', '##a', '##ts']
cap → ['c', '##a', '##p']


In [6]:
vocab = {
    "c",
    "##a",
    "##t",
    "##s",
    "##p",
    "ca",
    "cat",
    "##ts"
}

def tokenize_word(word, vocab):
    tokens = []

    while len(word) > 0:
        found = False

        for i in range(len(word), 0, -1):
            part = word[:i]

            if len(tokens) > 0:
                part = "##" + part

            if part in vocab:
                tokens.append(part)
                word = word[i:]
                found = True
                break

        if not found:
            return ["[UNK]"]

    return tokens


result = tokenize_word("cats", vocab)

print("Input:", "cats")
print("Tokens:", result)

Input: cats
Tokens: ['cat', '##s']


In [7]:
vocab_ids = {
    "[UNK]": 0,
    "c": 1,
    "##a": 2,
    "##t": 3,
    "##s": 4,
    "##p": 5,
    "ca": 6,
    "cat": 7,
    "##ts": 8
}

tokens = tokenize_word("cats", vocab)

ids = [vocab_ids[token] for token in tokens]

print("Tokens:", tokens)
print("Token IDs:", ids)

Tokens: ['cat', '##s']
Token IDs: [7, 4]
